<a href="https://colab.research.google.com/github/vcasadei/Redes-Neurais-CESAR-School/blob/2026.01-A/2%20-%20Regress%C3%A3o%20Linear/2.3_regressao_linear_tensores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 2.3 — Regressão Linear com tensores do PyTorch

## Objetivos

- Reescrever o notebook [2.2_regressao_linear_numpy.ipynb](2.2_regressao_linear_numpy.ipynb) usando tensores do PyTorch no lugar de arrays NumPy.
- Conhecer as operações de tensores equivalentes às do NumPy: criação, concatenação, transposição e produto de matrizes.
- Manter o gradiente calculado à mão, com a mesma fórmula do 2.2. O cálculo automático do gradiente (autograd) fica para o notebook 2.4.

O problema é o mesmo: prever o comprimento da pétala da íris (versicolor) a partir do comprimento da sépala, com a reta $\hat{y} = w_0 + w_1 x$ (nos slides, $h_\theta(x) = \theta_0 + \theta_1 x$, com $w_0 = \theta_0$ = bias e $w_1 = \theta_1$ = peso).

Um **tensor** é o array multidimensional do PyTorch: como o `ndarray` do NumPy, ele tem `shape` e `dtype`, mas também pode rodar em GPU e registrar operações para o cálculo automático de gradientes (que usaremos a partir do 2.4).

## Importação das bibliotecas

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch
from sklearn.datasets import load_iris

torch.manual_seed(1234)

def print_assert(r):
    if r:
        return 'OK!'
    else:
        return 'Incorreto!'

## Dados

### Leitura dos dados

`torch.from_numpy(...)` cria um tensor a partir de um array NumPy; `.float()` converte para `float32`, o tipo padrão do PyTorch para parâmetros e dados (o NumPy usa `float64`).

In [ ]:
iris = load_iris()
data = iris.data[iris.target == 1, ::2]  # versicolor; colunas 0 e 2: comprimento das sépalas e das pétalas

x_in = data[:, 0:1]
y_in = data[:, 1:2]

x_train = torch.from_numpy(x_in).float()   # tensor float32, shape (50, 1)
y_train = torch.from_numpy(y_in).float()

n_samples = x_train.size(0)   # .size(0): tamanho da dimensão 0 (número de amostras), igual a x_train.shape[0]
print('x_train.shape:', x_train.shape, x_train.dtype, type(x_train))
print('y_train.shape:', y_train.shape, y_train.dtype, type(y_train))

print('x_train[:5]:\n', x_train[:5])
print('y_train[:5]:\n', y_train[:5])

iris_pd = pd.DataFrame(x_in, columns=['x_in'])
iris_pd['y_in'] = y_in
iris_pd.head()

### Normalização dos dados

Normalização min-max para $[0, 1]$, como no 2.2. As operações `-=` e `/=` modificam o tensor no lugar (in-place). `.numpy()` converte o tensor de volta em array NumPy para o matplotlib.

In [ ]:
x_train -= x_train.min()
x_train /= x_train.max()
y_train -= y_train.min()
y_train /= y_train.max()

plt.scatter(x_train.numpy(), y_train.numpy())
plt.xlabel('Comprimento normalizado das sépalas')
plt.ylabel('Comprimento normalizado das pétalas')
plt.show()

### Exercício 1 — Criação do `x_train_bias`

Crie o tensor `x_train_bias`: uma matriz com 1s na primeira coluna e `x_train` na segunda coluna, shape $(M, 2) = (50, 2)$. É o equivalente de `np.hstack([np.ones((n_samples, 1)), x_train])` do notebook 2.2.

As células seguintes usam este resultado: resolva o exercício e rode a célula de verificação antes de continuar.

Dica: você pode precisar das funções [torch.cat](https://pytorch.org/docs/stable/generated/torch.cat.html) e [torch.ones](https://pytorch.org/docs/stable/generated/torch.ones.html).

In [ ]:
# TODO: escreva sua solução aqui
print('x_train_bias.shape:', x_train_bias.shape)

In [ ]:
# Verificação do exercício - Execute para testar sua implementação

print('x_train_bias é um tensor float32: {}'.format(print_assert(
    isinstance(x_train_bias, torch.Tensor) and x_train_bias.dtype == torch.float32
)))

print('tamanho do x_train_bias: {}'.format(print_assert(
    x_train_bias.size() == torch.Size([50, 2])
)))

print('primeira coluna é só de uns: {}'.format(print_assert(
    (x_train_bias[:, 0] - torch.ones(n_samples)).abs().sum() == 0
)))

print('segunda coluna é igual a x_train: {}'.format(print_assert(
    (x_train_bias[:, 1] - x_train.reshape(50)).abs().sum() == 0
)))

## Modelo

### Definição da classe

Mesma classe do 2.2, com tensores. `torch.empty(n_out, n_in)` reserva um tensor de shape $(1, 2)$ sem inicializar os valores; `.uniform_(-0.1, 0.1)` preenche-o no lugar (o `_` final indica operação in-place) com valores aleatórios uniformes. `torch.t(w)` (ou `w.t()`) é a transposta e `matmul` é o produto de matrizes.

Os parâmetros são um tensor comum, sem `requires_grad`: neste notebook o gradiente é calculado à mão, então o PyTorch não precisa registrar as operações.

In [ ]:
class Net():
    def __init__(self, n_in, n_out):
        self.w = torch.empty(n_out, n_in).uniform_(-0.1, 0.1)  # inicialização dos parâmetros, shape (1, 2)

    def forward(self, x_bias):
        return x_bias.matmul(torch.t(self.w))   # (M, 2) x (2, 1) -> (M, 1)

### Instanciando a rede

In [ ]:
model = Net(2, 1)
print('model.w:', model.w, 'shape:', model.w.shape)

### Testando a predição da rede

In [ ]:
model.forward(torch.ones((4, 2)))   # 4 amostras, 2 entradas (1 + x0) -> shape (4, 1)

## Treinamento

### Exercício 2 — Laço de treinamento

Refaça o código de treinamento da rede usando somente tensores do PyTorch. Siga o mesmo procedimento do notebook [2.2_regressao_linear_numpy.ipynb](2.2_regressao_linear_numpy.ipynb); as linhas NumPy estão nos comentários do laço. As fórmulas são as mesmas:

$$ J(\mathbf{w}) = \frac{1}{M} \sum_{i=0}^{M-1} (\hat{y}_i - y_i)^2, \qquad \nabla_{\mathbf{w}} J = \frac{2}{M}\, \mathbf{X}^T (\mathbf{X} \mathbf{w}^T - \mathbf{y}), \qquad \mathbf{w} \leftarrow \mathbf{w} - \eta\, (\nabla_{\mathbf{w}} J)^T $$

onde $\mathbf{X}$ é `x_train_bias` $(M, 2)$, $\mathbf{y}$ é `y_train` $(M, 1)$, $\mathbf{w}$ é `model.w` $(1, 2)$, $M$ é `n_samples` e $\eta$ é `lr`.

As células seguintes (avaliação) usam o modelo treinado aqui.

Correspondências úteis NumPy → PyTorch (consulte a [documentação do PyTorch](https://pytorch.org/docs/stable/torch.html)):

| NumPy | PyTorch |
|---|---|
| `a.T` | `a.t()` ou `torch.t(a)` |
| `a.dot(b)` | `a.matmul(b)` ou `torch.matmul(a, b)` |
| `np.square(a).mean()` | `(a ** 2).mean()` ou `torch.mean(a ** 2)` |
| `np.hstack` | `torch.cat(..., dim=1)` |

Para imprimir o custo, use `loss.item()`: converte um tensor de um único elemento num número Python.

In [ ]:
n_epochs = 100
lr = 0.5
for epoch in range(n_epochs):

    # forward - predição
    y_pred = model.forward(x_train_bias)

    # cálculo da função de custo (MSE)
    # NumPy: loss = np.square(y_pred - y_train).mean()
    # TODO: escreva sua solução aqui

    # cálculo do gradiente pelas derivadas parciais
    # NumPy: w_grad = (2.0/n_samples) * (x_train_bias.T).dot(y_pred - y_train)
    # TODO: escreva sua solução aqui

    # gradiente descendente
    # NumPy: model.w = model.w - lr * w_grad.T
    # TODO: escreva sua solução aqui

    # verbose
    if (epoch + 1) % 20 == 0:
        print('Epoch[{}/{}], loss: {:.6f}'
              .format(epoch + 1, n_epochs, loss.item()))

## Avaliação

### Predição

In [ ]:
y_pred = model.forward(x_train_bias)
plt.plot(x_train.numpy(), y_train.numpy(), 'ro', label='Dados originais')
plt.plot(x_train.numpy(), y_pred.numpy(), 'kx-', label='Reta ajustada')
plt.xlabel('Comprimento normalizado das sépalas')
plt.ylabel('Comprimento normalizado das pétalas')
plt.legend()
plt.show()

### Parâmetros treinados

In [ ]:
print(model.w)   # [w0 w1]

### Valor ótimo: solução analítica

Equação normal, como no 2.2: $\mathbf{w}^T = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$. `torch.linalg.inv` é a inversa de matriz (equivale a `np.linalg.inv`) e `mm` é o produto de duas matrizes 2D. O resultado deve ser próximo do `model.w` treinado e do obtido no 2.2.

In [ ]:
x_bias = x_train_bias
y = y_train
w_opt = (torch.linalg.inv(x_bias.t().mm(x_bias)).mm(x_bias.t())).mm(y)
print(w_opt.t())

## Aprendizados

- Tensores do PyTorch fazem as mesmas operações que arrays NumPy: `torch.cat` ↔ `np.hstack`, `t()` ↔ `.T`, `matmul` ↔ `dot`, `torch.linalg.inv` ↔ `np.linalg.inv`.
- `torch.from_numpy(...).float()` converte os dados para tensores `float32`; `.numpy()` faz o caminho de volta e `.item()` extrai um escalar Python de um tensor de um elemento.
- O laço de treinamento e as fórmulas de custo e gradiente são idênticos aos do 2.2; só mudou a biblioteca.
- Até aqui o gradiente foi derivado e programado à mão. No 2.4, `requires_grad` e `backward()` calculam esse gradiente automaticamente.